# Sprint 3: SOTA Upgrade (Memory Efficient Pipeline)

Notebook ini mengimplementasikan perbaikan besar-besaran (Sprint 3) untuk menyamakan performa dengan project SOTA MSCNN-BiLSTM-AE.

## Key Improvements:
1. **Preprocessing Baru:** MinMax Scaler (0-1) yang bersih, fit hanya pada Benign Train.
2. **Clipping:** Menangani outlier ekstrim di data Test dengan clipping ke [0, 1].
3. **Memory Efficient:** Menggunakan `tf.data.Dataset` generator untuk streaming data (anti-OOM).
4. **Modular:** Menggunakan script terpisah untuk preprocessing yang konsisten.

In [ ]:
# @title Colab Bootstrap (with Drive Persistence)
# Jalankan cell ini jika di Google Colab untuk setup environment
from pathlib import Path
import os
import subprocess
import sys
import shutil

COLAB_BOOTSTRAP_ENABLE = True
COLAB_REPO_URL = "https://github.com/akwancakra/nids-cnn-lstm-autoencoder.git"
COLAB_BRANCH = "feat/sprint3-sota-upgrade"
COLAB_REPO_DIR = Path("/content/nids-cnn-lstm-autoencoder")

# Drive Persistence Config
COLAB_DRIVE_MOUNT = Path("/content/drive")
COLAB_PROJECT_DRIVE_ROOT = COLAB_DRIVE_MOUNT / "MyDrive/nids-cnn-lstm-autoencoder"

def _is_colab_runtime() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def _run_shell(cmd: list[str], cwd: Path | None = None) -> None:
    print(f"[CMD] {' '.join(cmd)}")
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)

def _ensure_symlink_dir(repo_path: Path, drive_target: Path) -> None:
    """Symlink repo folder to drive folder for persistence."""
    drive_target.mkdir(parents=True, exist_ok=True)
    
    if repo_path.is_symlink():
        if repo_path.resolve() == drive_target.resolve():
            print(f"[INFO] Symlink OK: {repo_path} -> {drive_target}")
            return
        repo_path.unlink()
    elif repo_path.exists():
        if any(repo_path.iterdir()):
            print(f"[WARN] Folder exists and not empty: {repo_path}. Skipping link.")
            return
        repo_path.rmdir()
        
    repo_path.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(str(drive_target), str(repo_path), target_is_directory=True)
    print(f"[INFO] Symlink created: {repo_path} -> {drive_target}")

if _is_colab_runtime() and COLAB_BOOTSTRAP_ENABLE:
    from google.colab import drive
    drive.mount(str(COLAB_DRIVE_MOUNT))
    
    if not COLAB_REPO_DIR.exists():
        _run_shell(["git", "clone", "-b", COLAB_BRANCH, COLAB_REPO_URL, str(COLAB_REPO_DIR)])
    else:
        _run_shell(["git", "fetch", "--all"], cwd=COLAB_REPO_DIR)
        _run_shell(["git", "checkout", COLAB_BRANCH], cwd=COLAB_REPO_DIR)
        _run_shell(["git", "pull", "origin", COLAB_BRANCH], cwd=COLAB_REPO_DIR)
        
    # PERSISTENCE: Link Sprint 3 Data & Models to Drive
    # 1. Processed Data
    _ensure_symlink_dir(
        COLAB_REPO_DIR / "data/research/sprint3_upgrade/processed",
        COLAB_PROJECT_DRIVE_ROOT / "data/research/sprint3_upgrade/processed"
    )
    # 2. Models & Results
    _ensure_symlink_dir(
        COLAB_REPO_DIR / "research/sprint3_upgrade/models",
        COLAB_PROJECT_DRIVE_ROOT / "research/sprint3_upgrade/models"
    )
    _ensure_symlink_dir(
        COLAB_REPO_DIR / "research/sprint3_upgrade/results",
        COLAB_PROJECT_DRIVE_ROOT / "research/sprint3_upgrade/results"
    )
        
    # Set Working Directory
    os.chdir(COLAB_REPO_DIR)
    print(f"[INFO] Working Directory set to: {os.getcwd()}")
else:
    print("Not in Colab or Bootstrap disabled.")

In [ ]:
# Setup Project Root & Imports
import os
import sys
from pathlib import Path
import yaml
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import glob
from tqdm.notebook import tqdm

# Deteksi Project Root (Fallback mechanism)
try:
    # Jika di Colab dan sudah chdir, getcwd() adalah root
    PROJECT_ROOT = Path(os.getcwd()).resolve()
    
    # Validasi sederhana: cek folder research ada atau tidak
    if not (PROJECT_ROOT / "research").exists():
        # Coba naik satu level (jika notebook dijalankan lokal dari folder notebooks/)
        if (PROJECT_ROOT.parent / "research").exists():
            PROJECT_ROOT = PROJECT_ROOT.parent
        else:
            # Fallback hardcoded untuk Colab jika chdir gagal
            PROJECT_ROOT = Path("/content/nids-cnn-lstm-autoencoder")
except Exception:
    PROJECT_ROOT = Path(".")

print(f"Project Root: {PROJECT_ROOT}")
sys.path.append(str(PROJECT_ROOT))

# Load Config Sprint 3
config_path = PROJECT_ROOT / "research/sprint3_upgrade/config/experiment_v1.yaml"

if config_path.exists():
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    print(f"Experiment: {config['experiment_name']}")
    print(f"Config Loaded: {config_path}")
else:
    print(f"ERROR: Config not found at {config_path}")
    print("Make sure you have cloned the repo and switched to the correct branch.")

## 1. Preprocessing (SOTA Style)
Jalankan script preprocessing baru yang menerapkan MinMax(0-1) dan Clipping.

In [ ]:
# Run Preprocessing Script (If data not exists)
processed_dir = PROJECT_ROOT / config['paths']['processed_data']
train_dir = processed_dir / "train"
test_dir = processed_dir / "test"

RUN_PREPROCESSING = False 

# --- AUTO DETECT & REUSE LOGIC ---
# Check if processed data already exists (locally or via symlink)
if train_dir.exists() and any(train_dir.iterdir()):
    print("[INFO] Processed data found! Skipping preprocessing.")
    RUN_PREPROCESSING = False

# Adjust Raw Paths for Colab Support
if os.path.exists("/content/drive/MyDrive/nids-data/raw"):
    print("[INFO] Colab Drive Raw Data Detected.")
    raw_train = Path("/content/drive/MyDrive/nids-data/raw/CIC-IDS2017")
    raw_test = Path("/content/drive/MyDrive/nids-data/raw/CSE-CIC-IDS2018")
else:
    # Local fallback or repo path
    raw_train = PROJECT_ROOT / config['paths']['raw_train']
    raw_test = PROJECT_ROOT / config['paths']['raw_test']
# -------------------------------------------------

if RUN_PREPROCESSING:
    print("Running Preprocessing Script... (This may take a while)")
    script_path = PROJECT_ROOT / "research/sprint3_upgrade/scripts/preprocess_sota.py"
    
    # Check raw data
    if not raw_train.exists():
        print(f"Error: Raw train path {raw_train} not found!")
        print("Please check config or mount drive correctly.")
    else:
        cmd = f'python "{script_path}" --raw_train "{raw_train}" --raw_test "{raw_test}" --output_dir "{processed_dir}" --seq_len {config["preprocessing"]["sequence_length"]} --stride {config["preprocessing"]["stride"]}'
        print(f"Executing: {cmd}")
        !{cmd}
else:
    print("Skipping Preprocessing (Data already exists).")
    print(f"Data Location: {processed_dir}")

## 2. Data Pipeline (Memory Efficient)
Menggunakan `tf.data.Dataset` generator agar hemat RAM.

In [ ]:
def npz_generator(data_dir):
    files = sorted(glob.glob(os.path.join(data_dir, "*.npz")))
    for f in files:
        try:
            with np.load(f, allow_pickle=True) as data:
                X = data['X'] if 'X' in data else (data['x'] if 'x' in data else None)
                if X is not None:
                    for i in range(len(X)):
                        yield X[i], X[i] # Target = Input (Autoencoder)
        except: pass

def create_dataset(data_dir, batch_size=256, shuffle=True, input_shape=(10, 77)):
    dataset = tf.data.Dataset.from_generator(
        lambda: npz_generator(data_dir),
        output_signature=(
            tf.TensorSpec(shape=input_shape, dtype=tf.float32),
            tf.TensorSpec(shape=input_shape, dtype=tf.float32)
        )
    )
    if shuffle:
        dataset = dataset.shuffle(buffer_size=10000)
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

def count_samples(data_dir):
    files = sorted(glob.glob(os.path.join(data_dir, "*.npz")))
    total = 0
    for f in files:
        try:
            with np.load(f, allow_pickle=True) as data:
                key = 'X' if 'X' in data else 'x'
                if key in data:
                    total += data[key].shape[0]
        except: pass
    return total

print("Pipeline Ready.")

## 3. Build & Train Model

In [ ]:
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Concatenate, Bidirectional, LSTM, Dropout, Flatten, Dense, RepeatVector, UpSampling1D
from tensorflow.keras.models import Model

def build_model(input_shape, encoding_dim=16):
    inputs = Input(shape=input_shape)
    # Encoder
    conv1 = Conv1D(32, 3, activation='relu', padding='same')(inputs)
    conv2 = Conv1D(32, 5, activation='relu', padding='same')(inputs)
    conv3 = Conv1D(32, 7, activation='relu', padding='same')(inputs)
    pool1 = MaxPooling1D(2)(conv1)
    pool2 = MaxPooling1D(2)(conv2)
    pool3 = MaxPooling1D(2)(conv3)
    concat = Concatenate()([pool1, pool2, pool3])
    bilstm1 = Bidirectional(LSTM(64, return_sequences=True))(concat)
    dropout1 = Dropout(0.2)(bilstm1)
    flatten = Flatten()(dropout1)
    encoded = Dense(encoding_dim, activation='relu')(flatten)
    
    # Decoder
    repeat = RepeatVector(input_shape[0] // 2)(encoded)
    bilstm2 = Bidirectional(LSTM(64, return_sequences=True))(repeat)
    dropout2 = Dropout(0.2)(bilstm2)
    upsample = UpSampling1D(2)(dropout2)
    decoded = Conv1D(input_shape[1], 3, activation='sigmoid', padding='same')(upsample)
    
    autoencoder = Model(inputs, decoded)
    autoencoder.compile(optimizer='adam', loss='mse')
    return autoencoder

# Setup Training
BATCH_SIZE = config['training']['batch_size']
total_train = count_samples(train_dir)
print(f"Total Train Samples: {total_train}")

if total_train > 0:
    full_ds = create_dataset(train_dir, batch_size=BATCH_SIZE)
    val_size = int(total_train * config['training']['validation_split'])
    train_size = total_train - val_size
    
    train_steps = train_size // BATCH_SIZE
    val_steps = val_size // BATCH_SIZE
    
    # Safety check
    if train_steps == 0: train_steps = 1
    if val_steps == 0: val_steps = 1
    
    train_ds = full_ds.take(train_steps)
    val_ds = full_ds.skip(train_steps).take(val_steps)
    
    # Build
    input_shape = (10, 77) # Default
    model = build_model(input_shape, encoding_dim=config['model']['encoding_dim'])
    model.summary()
    
    # Callbacks
    model_save_dir = PROJECT_ROOT / config['paths']['model_save_dir']
    model_save_dir.mkdir(parents=True, exist_ok=True)
    
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=config['training']['patience'], restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(str(model_save_dir / "best_model.h5"), save_best_only=True, monitor='val_loss')
    ]
    
    # Train
    history = model.fit(
        train_ds,
        epochs=config['training']['epochs'],
        steps_per_epoch=train_steps,
        validation_data=val_ds,
        validation_steps=val_steps,
        callbacks=callbacks,
        verbose=1
    )
    
    # Plot
    plt.plot(history.history['loss'], label='Train')
    plt.plot(history.history['val_loss'], label='Val')
    plt.legend()
    plt.show()

## 4. Evaluation
Hitung threshold dan evaluasi pada Test Set.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns

# 1. Calculate Threshold
print("Calculating Threshold...")
val_errors = []
for batch_x, _ in tqdm(val_ds, desc="Val Batches", total=val_steps):
    recon = model.predict(batch_x, verbose=0)
    mse = np.mean(np.square(batch_x - recon), axis=(1, 2))
    val_errors.extend(mse)
    
threshold = np.percentile(val_errors, config['thresholding']['percentile'])
print(f"Threshold (P{config['thresholding']['percentile']}): {threshold}")

# 2. Test Evaluation
print("Evaluating on Test Set...")
def test_gen():
    files = sorted(glob.glob(os.path.join(test_dir, "*.npz")))
    for f in files:
        try:
            with np.load(f, allow_pickle=True) as data:
                X = data['X'] if 'X' in data else (data['x'] if 'x' in data else None)
                y = data['y'] if 'y' in data else (data['Y'] if 'Y' in data else None)
                if X is not None:
                    for i in range(len(X)):
                        yield X[i], y[i]
        except: pass

test_ds = tf.data.Dataset.from_generator(
    test_gen,
    output_signature=(
        tf.TensorSpec(shape=input_shape, dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int32)
    )
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

all_errors = []
all_y = []

for batch_x, batch_y in tqdm(test_ds, desc="Test Batches"):
    recon = model.predict(batch_x, verbose=0)
    mse = np.mean(np.square(batch_x - recon), axis=(1, 2))
    all_errors.extend(mse)
    all_y.extend(batch_y.numpy())
    
y_pred = (np.array(all_errors) > threshold).astype(int)
y_true = np.array(all_y)

# Metrics
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision: {precision_score(y_true, y_pred):.4f}")
print(f"Recall: {recall_score(y_true, y_pred):.4f}")
print(f"F1-Score: {f1_score(y_true, y_pred):.4f}")

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.show()